# End of week 3 exercise Synthetic data generator
## AI-Powered Clinical Case Generator
Purpose:
In this notebook we create a tool for medical educators to generate realistic OSCE (Objective Structured Clinical Examination) patient cases. It combines synthetic data generation with Large Language Models (LLMs) to create detailed clinical scenarios.
How It Works:
Data Generation: It creates random patient profiles (age, sex, vitals, labs) based on specific disease profiles (e.g., Type 2 Diabetes, Pneumonia).
LLM Integration: It sends this structured data to an AI model (via OpenRouter) to write a realistic clinical narrative (History, Physical Exam, Assessment) without revealing the diagnosis.
Visualization: It displays the results in a Gradio dashboard with a structured table.

In [ ]:
import os
import random
import uuid
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise ValueError("OPENROUTER_API_KEY not found")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

In [ ]:
MODELS = [
    "qwen/qwen3-235b-a22b-thinking-2507",
    "openai/gpt-4o-mini",
    "anthropic/claude-3.5-sonnet",
    "meta-llama/llama-3.1-70b-instruct",
    "mistralai/mixtral-8x7b-instruct"
]

In [ ]:
disease_profiles = {
    "Type 2 Diabetes": {"symptoms": ["polyuria","polydipsia","fatigue","blurred vision"], "labs": {"glucose": (200,350), "hba1c": (7.5,11)}},
    "Pneumonia": {"symptoms": ["cough","fever","shortness of breath","chest pain"], "labs": {"wbc": (12000,20000),"oxygen_sat":(85,93)}},
    "Asthma": {"symptoms": ["wheezing","shortness of breath","chest tightness"], "labs": {"oxygen_sat":(88,96)}},
    "Hypertension": {"symptoms": ["headache","dizziness","blurred vision"], "labs": {"systolic_bp":(140,190)}},
    "Tuberculosis": {"symptoms": ["chronic cough","night sweats","weight loss","fever"], "labs": {"wbc":(9000,15000)}}
}


In [ ]:
patients_db = {}


def generate_patient(disease):
    age = random.randint(18,80)
    sex = random.choice(["Male","Female"])
    profile = disease_profiles[disease]
    symptoms = random.sample(profile["symptoms"],2)
    labs = {k: round(random.uniform(v[0],v[1]),1) for k,v in profile["labs"].items()}
    vitals = {"blood_pressure": f"{random.randint(120,170)}/{random.randint(80,105)}",
              "heart_rate": random.randint(70,110),
              "temperature": round(random.uniform(36.5,39.0),1)}
    patient_id = f"SP-{str(uuid.uuid4())[:6]}"
    patient = {"id": patient_id, "age": age, "sex": sex, "symptoms": symptoms,
               "labs": labs, "vitals": vitals, "disease": disease}
    patients_db[patient_id] = patient
    return patient

In [ ]:
def build_prompt(patient):
    return f"""
    You are a clinical educator creating realistic OSCE patient cases.

    Generate a detailed clinical narrative using this structured data.

    Patient:
    Age: {patient["age"]}
    Sex: {patient["sex"]}

    Symptoms:
    {patient["symptoms"]}

    Vitals:
    {patient["vitals"]}

    Labs:
    {patient["labs"]}

    Write sections:

    Chief Complaint
    History of Present Illness
    Past Medical History
    Physical Examination
    Lab Interpretation
    Assessment
    Teaching Question

    Do NOT explicitly reveal the diagnosis.
    """

In [ ]:
def llm(prompt, model):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role":"system","content":"You generate realistic clinical cases."},
                  {"role":"user","content":prompt}],
        temperature=0.7,
        max_tokens=900
    )
    return response.choices[0].message.content


In [ ]:
def create_cases(diseases, model, num_per_disease):
    rows = []
    for disease in diseases:
        for _ in range(num_per_disease):
            patient = generate_patient(disease)
            narrative = llm(build_prompt(patient), model) if len(rows) < 5 else "Clinical narrative not generated for performance."
            
            # Extract vitals
            blood_pressure = patient["vitals"].get("blood_pressure", "")
            heart_rate = patient["vitals"].get("heart_rate", "")
            temperature = patient["vitals"].get("temperature", "")
            
            # Extract labs (handle missing values with N/A)
            glucose = patient["labs"].get("glucose", "N/A")
            hba1c = patient["labs"].get("hba1c", "N/A")
            wbc = patient["labs"].get("wbc", "N/A")
            oxygen_sat = patient["labs"].get("oxygen_sat", "N/A")
            systolic_bp = patient["labs"].get("systolic_bp", "N/A")
            
            # Format lab values - show only if not N/A
            labs_display = []
            if glucose != "N/A":
                labs_display.append(f"Glucose: {glucose}")
            if hba1c != "N/A":
                labs_display.append(f"HbA1c: {hba1c}")
            if wbc != "N/A":
                labs_display.append(f"WBC: {wbc}")
            if oxygen_sat != "N/A":
                labs_display.append(f"O2 Sat: {oxygen_sat}")
            if systolic_bp != "N/A":
                labs_display.append(f"Systolic BP: {systolic_bp}")
            
            # labs_str = ", ".join(labs_display) if labs_display else "N/A"
            
            # Create row with individual columns
            row = [
                patient["id"],  # PatientID
                patient["sex"],  # Sex
                patient["age"],  # Age
                blood_pressure,  # Blood Pressure
                heart_rate,  # Heart Rate
                temperature,  # Temperature
                glucose if glucose != "N/A" else "",  # Glucose
                hba1c if hba1c != "N/A" else "",  # HbA1c
                narrative,  # Notes
                patient["disease"]  # Diagnosis
            ]
            rows.append(row)
    return rows

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("## 🏥 Patient Dashboard")

    with gr.Row():
        with gr.Column(scale=1):
            disease_select = gr.CheckboxGroup(
                choices=list(disease_profiles.keys()),
                label="Select Disease(s)"
            )
            model_select = gr.Dropdown(MODELS, value=MODELS[0], label="LLM Model")
            num_patients = gr.Slider(minimum=1, maximum=20, value=5, step=1, label="Number of Patients per Disease")
            generate_btn = gr.Button("Generate Patients")

        with gr.Column(scale=3):
            table = gr.Dataframe(
                headers=["PatientID", "Sex", "Age", "Blood Pressure", "Heart Rate", "Temperature", "Glucose", "HbA1c", "Notes", "Diagnosis"],
                datatype=["str", "str", "str", "str", "str", "str", "str", "str", "str", "str"],
                interactive=False,
                row_count=(1,100),
                wrap=True
            )

    generate_btn.click(
        create_cases,
        inputs=[disease_select, model_select, num_patients],
        outputs=[table]
    )

if __name__ == "__main__":
    demo.launch()